In [ ]:
# 导入 PyTorch 核心模块
import torch
import torch.nn as nn              # 神经网络层和损失函数
import torch.optim as optim        # 优化器（SGD, Adam等）

from torch.utils.data import DataLoader          # 批量数据加载器
from torchvision import datasets, transforms    # 常用数据集和图像预处理

import matplotlib.pyplot as plt    # 可视化（显示图像）

In [ ]:
# ---------------------- 1. 超参数设置 ----------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 64
learning_rate = 0.001
epochs = 5

In [ ]:
# ---------------------- 2. 数据预处理 ----------------------
# transforms.Compose 将多个预处理串联：转张量 + 归一化
transform = transforms.Compose([
    transforms.ToTensor(),                       # PIL Image -> Tensor, 并缩放到 [0,1]
    transforms.Normalize((0.1307,), (0.3081,))   # MNIST 的标准化参数：(mean, std)
])

# 下载并加载 MNIST 数据集
train_dataset = datasets.MNIST(
    root='./data',      # 数据存放路径
    train=True,         # 训练集
    download=True,      # 自动下载
    transform=transform
)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True        # 每个 epoch 打乱数据
)

In [ ]:
# ---------------------- 3. 定义 CNN 模型 ----------------------
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # 卷积层：1通道(灰度) -> 32通道，卷积核3x3
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)            # 2x2 最大池化，尺寸减半
        self.dropout = nn.Dropout(0.25)           # 防止过拟合
        # 全连接层：MNIST 28x28 经过两次池化后变为 7x7，64通道 -> 7*7*64=3136
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)           # 输出10类（数字0-9）

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))  # [64, 32, 14, 14]
        x = self.pool(torch.relu(self.conv2(x)))  # [64, 64, 7, 7]
        x = x.view(x.size(0), -1)                 # 展平：[batch_size, 3136]
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)

In [ ]:
# ---------------------- 4. 损失函数与优化器 ----------------------
criterion = nn.CrossEntropyLoss()               # 交叉熵损失（内置 Softmax）
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# ---------------------- 5. 训练循环 ----------------------
loss_history = []

for epoch in range(epochs):
    model.train()  # 训练模式（启用 Dropout/BatchNorm）
    running_loss = 0.0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        # 前向传播
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 反向传播
        optimizer.zero_grad()   # 清空上一轮梯度
        loss.backward()         # 计算梯度
        optimizer.step()        # 更新参数
        
        running_loss += loss.item()
        
        if (batch_idx + 1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Step [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}')
    
    avg_loss = running_loss / len(train_loader)
    loss_history.append(avg_loss)

In [ ]:
# ---------------------- 6. 可视化 Loss 曲线 ----------------------
plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), loss_history, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Average Loss')
plt.title('Training Loss Curve')
plt.grid(True)
plt.show()

In [ ]:
# ---------------------- 7. 测试：查看一张预测结果 ----------------------
model.eval()  # 评估模式
with torch.no_grad():  # 不计算梯度，节省内存
    images, labels = next(iter(train_loader))
    images = images.to(device)
    outputs = model(images)
    predictions = outputs.argmax(dim=1)
    
    # 显示第一张图
    plt.imshow(images[0].cpu().squeeze(), cmap='gray')
    plt.title(f'Label: {labels[0].item()}, Pred: {predictions[0].item()}')
    plt.show()